# Velocity — divergence-free against unconstrained

Supports Supplementary Note 2. The recumbent fold of `01`, fitted with two
velocities that differ only in whether $\nabla\cdot\boldsymbol v$ can be nonzero:

- **Euler**: $\boldsymbol v=\nabla\alpha\times\hat{\boldsymbol k}$ from one potential of
  $M$ modes, as in the paper, so $\nabla\cdot\boldsymbol v=0$ identically;
- **free**: $\boldsymbol v=(\phi_x,\phi_z)$, two independent Fourier series of $M/2$
  modes each, so the divergence is a free field.

Both have $3M$ learnable parameters and the paper's training: amplitudes from
zero by gradient descent with momentum, phases and biases by Adam, at the
main-text rates. The band (`BAND`) reaches shorter than the main text's: the
main-text band is for the composition of shears, which tightens folds that a
single stationary flow can only make with shorter modes. The free velocity cannot be split into shears, so
**both are integrated with RK4** (`N_STEPS` steps), and the integrator is not
what differs.

**Matching the scales.** A mode of the Euler velocity is a gradient of a
potential weighted by $\lVert\boldsymbol\omega\rVert^{-2}$, so its velocity gradient
per unit amplitude does not depend on the wavelength. A component of the free
velocity is a potential value, so the same holds with the weight
$\lVert\boldsymbol\omega\rVert^{-1}$ (`freq_damp=1`). Each free component has half the
modes, so a scale of $1/\sqrt2$ gives it the same $1/\sqrt M$ prefactor. Both
are energy-normalised.

Each of `N_SEEDS` bases (the seeds of `s01`, first the main-text seed) is fitted
both ways, so comparisons are paired within a seed. Recorded per fit:

- the loss every `EVERY` iterations;
- interior extrema of $\psi_p$, and the IoU and contact distance of the units
  against the truth;
- the **restored gap** between the two horizons, $|\psi_p(H0)-\psi_p(H1)|$; the
  true gap is 130 m;
- $J=\det\mathbf F$ and the strain energy $\langle\sum_k(\ln\lambda_k)^2\rangle$ (its 90th
  percentile too) over a Poisson-disc sample of the grid;
- $|\nabla\cdot\boldsymbol v|$ over the same points, and the largest
  $\lVert\mathbf D\tilde{\boldsymbol\varphi}_h-\mathbf I\rVert_2$, with $\tilde{\boldsymbol\varphi}_h$ one RK4 step of $h=1/N$
  in either direction: below 1, each step is injective, so the map is a
  diffeomorphism.

For the main-text seed, also the restored elevation and $\ln J$ on the grid and
the contacts. Writes `results/velocity.npz`, read by `s02b_velocity_figures`.
Runs in `curlew_dev`.

## Setup

In [1]:
%matplotlib inline
import json
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
import torch.nn as nn

sys.path.insert(0, str(Path("..").resolve()))                      # publication/: diagnostics
sys.path.insert(0, str(Path("../01_recumbent_fold").resolve()))    # recumbent_inputs, synthetic_fold
import diagnostics
import recumbent_inputs

import curlew
from curlew.core import CSet, HSet
from curlew.geometry import Grid
from curlew.geology import restore
from curlew.kinematics.euler import euler_velocity
from curlew.fields.series import FSF

curlew.device = "cuda" if torch.cuda.is_available() else "cpu"
curlew.dtype = torch.float32
curlew.default_dim = 2

FOLD_DIR = Path("../01_recumbent_fold")
DATA_DIR = FOLD_DIR / "data"
P = json.loads((FOLD_DIR / "results/synthetic/fit_settings.json").read_text())   # written by 01b
RES_DIR = Path("./results")
RES_DIR.mkdir(parents=True, exist_ok=True)

## Data, grid and truth

As in `s01c`.

In [2]:
section = recumbent_inputs.load_section("synthetic", DATA_DIR)
horizon_names, horizons = section["horizon_names"], section["horizons"]
gp, gv, origin = section["gp"], section["gv"], section["origin"]
anchor = torch.zeros(2, dtype=curlew.dtype, device=curlew.device)
C = CSet(gp=gp, gv=gv, eq=horizons, iq=(P["iq_samples"], [(horizons[0], horizons[1], "<")]))

G = Grid((1200, 1200), step=(5.0, 5.0), center=np.zeros(2), sampleArgs=dict(N=128))
domaxes = G.tformed_axes
grd = torch.tensor(G.coords(), dtype=curlew.dtype, device=curlew.device)
unit_true = recumbent_inputs.true_units(domaxes[0], domaxes[1], origin, DATA_DIR)
reference = recumbent_inputs.truth_reference(origin, DATA_DIR)
strain_points = np.asarray(G.sample(poissonDisk=(20.0, 100_000, 0)))
print(f"{len(strain_points)} strain points; horizons {horizon_names}")

truth = np.load(DATA_DIR / "synthetic/truth.npz")
true_levels = [np.median(diagnostics.sample_regular_grid(truth["x"], truth["z"], truth["restored_height"], truth[k]))
               for k in ("contact_rock2", "contact_rock1")]
TRUE_GAP = abs(true_levels[0] - true_levels[1])
print(f"true restored gap {TRUE_GAP:.1f} m")

2580 strain points; horizons ['H0', 'H1']
true restored gap 130.0 m


## The two velocities

`FreeVelocity` stacks two Fourier series as the components. Its Jacobian has
the components' gradients as rows (not Hessians), which is all RK4 needs.

In [3]:
N_STEPS = 8              # RK4 steps over unit time, for both velocities
EPOCHS = P["epochs_ldi"]
EVERY = 25
BAND = (500.0, 4000.0)   # shorter than the main-text 1-5 km: a stationary flow, unlike the
                         # composition of shears, cannot tighten folds beyond its own modes


class FreeVelocity(nn.Module):
    """Unconstrained 2-D velocity: v = (phi_x, phi_z) from two independent Fourier series."""

    def __init__(self, vx: FSF, vz: FSF):
        super().__init__()
        self.vx, self.vz, self.dim = vx, vz, 2

    def forward(self, x, w=None, t=None):
        return torch.stack((self.vx.evaluate_raw(x, w), self.vz.evaluate_raw(x, w)), dim=-1)

    def forward_and_jacobian(self, x, w=None, t=None):
        jv = torch.stack((self.vx.gradient_raw(x, w), self.vz.gradient_raw(x, w)), dim=1)
        return self.forward(x, w), jv


def make_velocity(variant, seed):
    common = dict(n_directions=P["n_directions"], length_scale_range=BAND, activation="cubic",
                  energy_normalised=True)
    if variant == "euler":
        return euler_velocity(2, n_features=P["n_features"], seed=int(seed), stream="alpha", **common)
    # Each component: half the modes, weight |w|^-1 (a value, not a gradient), and scale
    # 1/sqrt(2) so that both velocities share the 1/sqrt(M) prefactor.
    comp = lambda name, s: FSF(name, H=HSet(), input_dim=2, rff_features=P["n_features"] // 2, seed=int(s),
                               freq_damp=1.0, scale=2 ** -0.5, xavier_amplitudes=False, **common)
    return FreeVelocity(comp("vx", seed), comp("vz", seed + 1))


def make_ldi(variant, seed):
    velocity = make_velocity(variant, seed)
    optimizer = {
        "amplitude": (torch.optim.SGD, dict(lr=P["lr_sgd"], momentum=P["amp_momentum"])),
        "shape": (torch.optim.Adam, dict(lr=P["lr_adam"])),
    }
    event = restore(
        f"s02_{variant}", C=C, H=HSet().zero(**P["weights"]), velocity=velocity,
        n_steps=N_STEPS, integrator="rk4", depth_axis=1,
        init_std=0.0, sigma_floor=0.0, signed_normals=True,
        learning_rate=P["lr_adam"], optimizer=optimizer, pin=anchor,
    )
    for name, h in zip(reversed(horizon_names), reversed(horizons)):
        event.addIsosurface(name, seed=h)
    return curlew.GeoModel([event]), event, velocity

## Fitting and measuring

In [4]:
def rk4_step_gradient(velocity, x, h):
    # ||D Phi_h - I||_2 at x for one RK4 step of size h: the stage Jacobians carried through
    # the four stages, D Phi_h = I + h/6 (J1 + 2 J2 + 2 J3 + J4).
    eye = torch.eye(2, dtype=x.dtype, device=x.device)
    k1, J1 = velocity.forward_and_jacobian(x)
    k2, a2 = velocity.forward_and_jacobian(x + 0.5 * h * k1)
    J2 = a2 @ (eye + 0.5 * h * J1)
    k3, a3 = velocity.forward_and_jacobian(x + 0.5 * h * k2)
    J3 = a3 @ (eye + 0.5 * h * J2)
    k4, a4 = velocity.forward_and_jacobian(x + h * k3)
    J4 = a4 @ (eye + h * J3)
    return torch.linalg.matrix_norm(h / 6.0 * (J1 + 2.0 * J2 + 2.0 * J3 + J4), ord=2)


def velocity_measures(velocity):
    # |div v| at the strain points, and the largest ||D Phi_h - I||_2 of one RK4 step from them,
    # forwards and backwards (the restoration integrates the inverse flow).
    x = torch.as_tensor(strain_points, dtype=curlew.dtype, device=curlew.device)
    with torch.no_grad():
        _, jv = velocity.forward_and_jacobian(x)
        div = jv.diagonal(dim1=-2, dim2=-1).sum(-1).abs()
        step = max(float(rk4_step_gradient(velocity, x, s / N_STEPS).max()) for s in (1.0, -1.0))
    return div.cpu().numpy(), step


def train(variant, seed, keep_grids=False):
    torch.manual_seed(405)
    model, event, velocity = make_ldi(variant, seed)
    log = dict(iteration=[], loss=[])

    def record(epoch, epochs):
        if epoch % EVERY == 0:
            with torch.no_grad():
                log["iteration"].append(epoch)
                log["loss"].append(float(event.loss().total()))

    t0 = perf_counter()
    model.prefit(epochs=EPOCHS, vb=False, early_stop=None, best=True, schedule=record)
    record(0, 0)
    log["iteration"][-1] = EPOCHS

    d = diagnostics.evaluate_ensemble([model], [event], grd, G.shape, domaxes[0], domaxes[1], grid=G)
    units = recumbent_inputs.match_units(d["lithology"][0], unit_true)
    s = recumbent_inputs.score_against_truth(units, domaxes[0], domaxes[1], unit_true, reference)
    levels = diagnostics.horizon_isovalues(domaxes[0], domaxes[1], d["scalar"][0], horizons)
    m = diagnostics.strain_measures(event, strain_points)
    hencky = np.sum(np.log(m["stretches"]) ** 2, axis=1)
    div, step = velocity_measures(velocity)
    out = {k: np.asarray(v) for k, v in log.items()}
    out.update(
        extrema=int(d["extrema"].sum()), iou=np.asarray(s["iou"]), contact_distance=np.asarray(s["contact_distance"]),
        gap=abs(levels[0] - levels[1]), log_j=np.percentile(np.log(m["det"]), [5, 50, 95]),
        strain=float(hencky.mean()), strain_p90=float(np.percentile(hencky, 90)),
        div=np.percentile(div, [50, 99]), step=step,
        contacts=diagnostics.ensemble_contacts(d["scalar"], domaxes[0], domaxes[1], horizons)[0],
    )
    if keep_grids:
        out["psi"] = d["scalar"][0].astype(np.float32)
        out["log_j_grid"] = np.log(diagnostics.strain_measures(event, G.coords())["det"]).reshape(G.shape).astype(np.float32)
    print(f"seed {seed}, {variant}: {perf_counter() - t0:.0f} s, loss {out['loss'][-1]:.3g}, extrema {out['extrema']}, "
          f"IoU {np.round(out['iou'], 2)}, gap {out['gap']:.0f} m (true {TRUE_GAP:.0f}), "
          f"ln J 5/50/95% {np.round(out['log_j'], 3)}, strain {out['strain']:.3f}, "
          f"|div v| median {out['div'][0]:.2g}, max ||D Phi_h - I|| {step:.2f}")
    return out

## Fits

In [5]:
N_SEEDS = 5
seed_rng = np.random.default_rng(P["fourier_seed"] + 91)        # as 01d, s01b and s01c
SEEDS = [int(P["fourier_seed"])]
while len(SEEDS) < N_SEEDS:
    candidate = int(seed_rng.integers(0, 2**31 - 1))
    if candidate not in SEEDS:
        SEEDS.append(candidate)

VARIANTS = ("euler", "free")
runs = {v: [train(v, seed, keep_grids=(k == 0)) for k, seed in enumerate(SEEDS)] for v in VARIANTS}

seed 42, euler: 30 s, loss 8.88e-05, extrema 0, IoU [0.9  0.77 0.95], gap 167 m (true 130), ln J 5/50/95% [-0.  0.  0.], strain 0.853, |div v| median 0, max ||D Phi_h - I|| 0.59
seed 1123574222, euler: 27 s, loss 7.94e-05, extrema 0, IoU [0.9  0.77 0.95], gap 164 m (true 130), ln J 5/50/95% [-0.  0.  0.], strain 0.857, |div v| median 0, max ||D Phi_h - I|| 0.59
seed 2094121401, euler: 27 s, loss 0.000104, extrema 0, IoU [0.9  0.77 0.95], gap 168 m (true 130), ln J 5/50/95% [-0.  0.  0.], strain 0.833, |div v| median 0, max ||D Phi_h - I|| 0.58
seed 86309145, euler: 27 s, loss 5.93e-05, extrema 0, IoU [0.9  0.78 0.95], gap 165 m (true 130), ln J 5/50/95% [-0.  0.  0.], strain 0.859, |div v| median 0, max ||D Phi_h - I|| 0.58
seed 1099952541, euler: 27 s, loss 7.18e-05, extrema 0, IoU [0.9  0.78 0.95], gap 164 m (true 130), ln J 5/50/95% [-0.  0.  0.], strain 0.858, |div v| median 0, max ||D Phi_h - I|| 0.56
seed 42, free: 57 s, loss 0.000406, extrema 0, IoU [0.8  0.56 0.85], gap 55 m (t

## Save

In [7]:
SCALARS = ("iteration", "loss", "extrema", "iou", "contact_distance", "gap", "log_j", "strain", "strain_p90", "div", "step")
arrays = {"variants": np.array(VARIANTS), "seeds": np.array(SEEDS), "true_gap": TRUE_GAP, "n_steps": N_STEPS, "band": np.array(BAND),
          "window": np.array(reference["window"]), "x": domaxes[0], "z": domaxes[1]}
for v, rs in runs.items():
    for key in SCALARS:
        arrays[f"{v}_{key}"] = np.stack([np.asarray(r[key]) for r in rs])       # seed x ...
    arrays[f"{v}_psi"], arrays[f"{v}_log_j_grid"] = rs[0]["psi"], rs[0]["log_j_grid"]   # main-text seed
arrays.update(diagnostics.pack_contacts([runs[v][0]["contacts"] for v in VARIANTS]))
np.savez(RES_DIR / "velocity.npz", **arrays)
print("wrote", RES_DIR / "velocity.npz")

for key in ("gap", "strain"):
    e, f = arrays[f"euler_{key}"], arrays[f"free_{key}"]
    print(f"{key}: Euler median {np.median(e):.3g}, free median {np.median(f):.3g}")

wrote results/velocity.npz
gap: Euler median 165, free median 55.1
strain: Euler median 0.857, free median 2.13
